In [1]:
import requests
from typing import Dict
import pandas as pd
from dotenv import load_dotenv
import os
import json
from datetime import datetime

In [2]:
BASE_URL = "http://localhost:8000"

In [3]:
MAX_ID_FILE = "max_ids.json"


def load_max_ids():
    if not os.path.exists(MAX_ID_FILE):
        with open(MAX_ID_FILE, "w") as f:
            json.dump({}, f)
        return {}

    with open(MAX_ID_FILE, "r") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return {}


def save_max_ids(max_ids: Dict[str, int]):
    with open(MAX_ID_FILE, "w") as f:
        json.dump(max_ids, f, indent=4)


def fetch_all_pages(endpoint: str, page_size: int = 10000):
    all_data = []
    page = 1

    max_ids = load_max_ids()

    last_max_id = max_ids.get(endpoint)

    while True:
        params = {
            "page_size": page_size,
        }

        if last_max_id is not None:
            params["last_max_id"] = last_max_id

        response = requests.get(
            f"{BASE_URL}{endpoint}",
            params=params,
            timeout=120
        )

        if response.status_code != 200:
            print(f"Error {response.status_code}: {response.text}")
            break

        result = response.json()
        items = result.get("data", [])

        if not items:
            break

        all_data.extend(items)
        print(f"Fetched page {page} → {len(items)} records")

        last_item = items[-1]

        if "id" in last_item:
            last_max_id = last_item["id"]
        elif "user_id" in last_item:
            last_max_id = last_item["user_id"]

        if len(items) < page_size:
            break

        page += 1

    if last_max_id is not None:
        max_ids[endpoint] = last_max_id
        save_max_ids(max_ids)

    df = pd.DataFrame(all_data)
    print(f"Total records fetched: {len(all_data)}")
    return df

In [ ]:
from io import BytesIO
import pandas as pd
from azure.storage.blob import BlobServiceClient


class AzureParquetUploader:
    def __init__(self, connection_string: str, container_name: str):
        self.blob_service_client = BlobServiceClient.from_connection_string(
            connection_string,
            max_single_put_size=4 * 1024 * 1024,
            max_block_size=4 * 1024 * 1024,
        )

        self.container_name = container_name

        container_client = self.blob_service_client.get_container_client(
            container_name
        )

        try:
            container_client.create_container()
        except Exception:
            pass


    def upload_df(
        self,
        df: pd.DataFrame,
        blob_name: str,
        timeout: int = 600,
        max_concurrency: int = 4,
    ):

        if df.empty:
            print(f"Skipping empty DataFrame → {blob_name}")
            return


        parquet_buffer = BytesIO()

        df.to_parquet(
            parquet_buffer,
            index=False,
            engine="pyarrow",
        )

        size_mb = parquet_buffer.tell() / 1024 / 1024
        parquet_buffer.seek(0)

        blob_client = self.blob_service_client.get_blob_client(
            container=self.container_name,
            blob=blob_name,
        )

        for attempt in range(3):
            try:
                blob_client.upload_blob(
                    parquet_buffer,
                    overwrite=True,
                    timeout=timeout,
                    max_concurrency=max_concurrency,
                )

                print(f"Uploaded ({size_mb:.1f} MB) → {blob_name}")
                return

            except Exception as e:
                print(
                    f"Upload failed (attempt {attempt + 1}/3) → {blob_name}: {e}"
                )
                parquet_buffer.seek(0)

        raise RuntimeError(f"Failed after 3 attempts → {blob_name}")

In [ ]:
load_dotenv()
CONNECTION_STRING = os.getenv("CONNECTION_STRING")
CONTAINER_NAME = os.getenv("CONTAINER_NAME")
uploader = AzureParquetUploader(connection_string = CONNECTION_STRING, container_name = CONTAINER_NAME)

date_folder = datetime.utcnow().strftime("%Y-%m-%d")

students = fetch_all_pages("/export_students")
courses = fetch_all_pages("/export_courses")
enrollments = fetch_all_pages("/export_enrollments")
attendance = fetch_all_pages("/export_attendance")
course_offerings = fetch_all_pages("/export_course_offerings")
department = fetch_all_pages("/export_department")
academic_semester = fetch_all_pages("/export_academic_semester")
tickets = fetch_all_pages("/export_tickets")
exams = fetch_all_pages("/export_exams")
quiz_submission = fetch_all_pages("/export_quiz_submissions")
exam_results = fetch_all_pages("/export_exam_results")
professors = fetch_all_pages("/export_professors")
quizzes = fetch_all_pages("/export_quizzes")


uploader.upload_df(students, f"bronze/students/{date_folder}/students.parquet")
uploader.upload_df(courses, f"bronze/courses/{date_folder}/courses.parquet")
uploader.upload_df(enrollments, f"bronze/enrollments/{date_folder}/enrollments.parquet")
uploader.upload_df(attendance, f"bronze/attendance/{date_folder}/attendance.parquet")
uploader.upload_df(course_offerings, f"bronze/course_offerings/{date_folder}/course_offerings.parquet")
uploader.upload_df(department, f"bronze/department/{date_folder}/department.parquet")
uploader.upload_df(academic_semester, f"bronze/academic_semester/{date_folder}/academic_semester.parquet")
uploader.upload_df(tickets, f"bronze/tickets/{date_folder}/tickets.parquet")
uploader.upload_df(exams, f"bronze/exams/{date_folder}/exams.parquet")
uploader.upload_df(quiz_submission, f"bronze/quiz_submission/{date_folder}/quiz_submission.parquet")
uploader.upload_df(exam_results, f"bronze/exam_results/{date_folder}/exam_results.parquet")
uploader.upload_df(professors, f"bronze/professors/{date_folder}/professors.parquet")
uploader.upload_df(quizzes, f"bronze/quizzes/{date_folder}/quizzes.parquet")

NameError: name 'load_dotenv' is not defined